# Task 1.2 — Key Assumptions (8 marks)

Below are three key assumptions that the Copula Mixture Model (CM) for dependency-seeking clustering depends on. Each assumption is specific to the proposed contribution in the paper rather than a generic machine learning assumption.

---

## Assumption 1: Continuous Marginal Densities Must Exist

**Assumption:** The method assumes that every dimension of both views possesses a *continuous* marginal distribution with a well-defined density function $f_j(x_j)$. This is required for the copula density decomposition (Equation 8) and for the copula transformation $\tilde{x}_j = \Phi^{-1}(F_j(x_j))$ to be well-defined and injective.

**Why the method needs it:** The entire meta-Gaussian density formula $f(x) = |P|^{-1/2} \exp\left(-\frac{1}{2} \tilde{x}^T (P^{-1} - I) \tilde{x}\right) \prod_j f_j(x_j)$ (Eq. 8) decomposes the joint density into a copula part and a marginal part. If any marginal is discrete or has point masses, $F_j$ has jumps making $\Phi^{-1}(F_j(\cdot))$ discontinuous, and the copula density $c(\cdot)$ in Equation 7 does not exist in closed form. The MCMC posterior updates in Algorithm 2 (Step 2) also sample the latent normal scores $\tilde{X}_j$ using the meta-Gaussian density, which requires the continuous density decomposition.

**Violation scenario:** Consider a dataset where one view contains *count data* (e.g., number of customer purchases per month — Poisson distributed) or *binary/categorical variables* (e.g., presence/absence of gene mutations). These are discrete distributions with point masses, so the probability integral transform produces ties (many observations map to the same $F_j$ value), the copula is no longer unique (Sklar's theorem guarantees uniqueness only for continuous margins), and the transformation to normal scores becomes ill-defined.

**Paper reference:** Section 3 — Theorem 1 (Sklar) states "if the margins are continuous, then this copula is unique"; Section 4.1 specifies "the only restriction being that a density must exist"; Equation 6 defines $f(x_1, \ldots, x_d) = c(F_1(x_1), \ldots, F_d(x_d)) \prod_j f_j(x_j)$ which requires continuous margins for $c$ to be well-defined.

---

## Assumption 2: The Gaussian Copula Adequately Captures the Dependence Structure

**Assumption:** The method assumes that the dependence between dimensions *within* each view, after transforming to normal scores, is well-described by a *Gaussian copula* — i.e., the joint dependence is characterised entirely by pairwise linear correlations of the normal scores. This means the model captures monotonic dependence patterns but cannot represent *tail dependencies* (stronger dependence in extremes) or other non-linear dependence structures that a Gaussian copula cannot express.

**Why the method needs it:** The meta-Gaussian density (Eq. 8) factorises the copula contribution through the correlation matrix $P$, and all inference (Algorithm 2, Step 3) samples the intra-view covariance matrices from an inverse-Wishart distribution, which is the conjugate prior for a multivariate normal. If the true intra-view dependence were better described by, say, a Clayton copula (which captures lower-tail dependence) or a Student-$t$ copula (which captures tail dependence in all directions), the Gaussian copula would systematically underestimate the strength of dependence in extreme observations, leading to incorrect cluster boundaries in the tails.

**Violation scenario:** In financial risk modelling, asset returns often exhibit *tail dependence* — large losses tend to co-occur more frequently than a Gaussian model predicts. If one view contains daily log-returns of correlated stocks, their joint behaviour in market crashes is more extreme than what a Gaussian copula can represent. The model would underestimate the clustering of extreme events, potentially assigning crash-period observations to incorrect clusters.

**Paper reference:** Section 3 ("Gaussian copulas constitute an important class of copulas"); Section 2, second paragraph (the model's parametrisation "using a correlation matrix covers many different dependence patterns ranging from independence to comonotonicity", but this is specifically for Gaussian copulas, not for general copulas); Section 4.2 — Algorithm 2 Step 3 uses the inverse-Wishart posterior, inherently tied to the Gaussian copula form.

---

## Assumption 3: Block-Diagonal Structure — Conditional Independence of Views Given Cluster Assignment

**Assumption:** The method assumes that, given the cluster assignment $Z$, the two views $X$ and $Y$ are *conditionally independent*. This is encoded via the block-diagonal covariance/correlation structure $\Sigma_z = \begin{pmatrix} \Sigma_{z}^x & 0 \\ 0 & \Sigma_{z}^y \end{pmatrix}$ (Eq. 4), which forces all cross-view zeros in the correlation matrix.

**Why the method needs it:** This is the structural core of dependency-seeking clustering — it compels the cluster structure to capture *all* dependencies between views. Without this constraint, the model could explain inter-view correlations within a single component (e.g., through off-diagonal entries in the full correlation matrix) without needing to separate observations into different clusters. The clustering would then reflect within-view structure rather than cross-view relationships. This block structure is what makes the method "dependency-seeking" rather than just a general-purpose clustering algorithm.

**Violation scenario:** Consider a medical study where View 1 contains physiological measurements (blood pressure, heart rate) and View 2 contains medication dosages. Suppose that within a single patient subgroup (one true cluster), there is a strong residual correlation between dosage and blood pressure *even after* accounting for group membership — for example, because within-group dose-response relationships vary continuously. The block-diagonal constraint would force the model to over-segment this group into many small clusters to explain the residual cross-view correlation, producing a fragmented, uninterpretable clustering.

**Paper reference:** Section 2, Equation (4) — the block structure of $\Sigma_z$; Section 3, last paragraph ("if $P$ has a block diagonal structure as in (1), the conditional independence of $X|Z$ and $Y|Z$... will be preserved in a meta-Gaussian model"); Section 5.2, Figure 7 and its discussion of product space vs. dependency-seeking clustering.